### Pandas

In [2]:
### Requisitos
import sys
import io
from decimal import Decimal
from math import pi

import pandas as pd
import numpy as np
import pyarrow as pa

pd.set_option("mode.copy_on_write", True)
pd.options.mode.copy_on_write = "warn"

#### Criação, Leitura e escrita de dados

##### Estruturas de Dados

1. **Series:** Unidimensional, é uma espécie de lista nomeada homogênea cujos valores podem ser de qualquer tipo. 
2. **Dataframes:** Bidimensional, pode ser pensado como um array de duas dimensões ou como uma tabela com linhas e colunas. As colunas de um dataframe são séries.

##### Criação

Dataframes podem ser criados a partir de um dicionário, onde cada Chave definirá o nome da coluna e o o valor representará as entradas do DataFrame. O método construtor da classe DataFrame aceita outros tipos de parâmetros como por exemplo iteradores, como listas e ndarray (numpy) desde que sejam estruturados ou homogêneos. 

 A partir da versão 2.0 o Pandas suporta nativamente os tipos do Arrow e em determinados casos, escolher entre os backends de tipos Arrow e Numpy faz muita diferença para um *Workflow* de dados. Veja mais em [PyArrow e Pandas](https://pandas.pydata.org/docs/user_guide/pyarrow.html)


In [8]:
df = pd.DataFrame(
    {
        "A": 1.0,
        "B": pd.Timestamp("20130102"),
        "C": pd.Series(1, index=list(range(4)), dtype="float32"),
        "D": np.array([3] * 4, dtype="int32"),
        "E": pd.Categorical(["test", "train", "test", "train"]),
        "F": "foo",
        "G": pd.Series(Decimal(pi**2), index=list(range(4)), dtype="float"),
    }

)

df

,A,B,C,D,E,F,G
0,1.0,2013-01-02,1.0,3,test,foo,9.869604
1,1.0,2013-01-02,1.0,3,train,foo,9.869604
2,1.0,2013-01-02,1.0,3,test,foo,9.869604
3,1.0,2013-01-02,1.0,3,train,foo,9.869604


In [5]:
df.dtypes

A          float64
B    datetime64[s]
C          float32
D            int32
E         category
F           object
G          float64
dtype: object

In [4]:

df.memory_usage()

Index     32
A         32
B         32
C         16
D         16
E        128
F         32
G         32
dtype: int64

In [7]:
decimal_type = pd.ArrowDtype(pa.decimal128(3, scale=2))

df_arrow = pd.DataFrame(
    {
        "A": 1.0,
        "B": pd.Timestamp("20130102"),
        "C": pd.Series(1, index=list(range(4)), dtype=pd.ArrowDtype(pa.float16())),
        "D": pd.array([3] * 4, dtype=pd.ArrowDtype(pa.uint16())),
        "E": pd.Categorical(["test", "train", "test", "train"]),
        "F": pa.array(["foo"]*4, type=pa.string()),
        "G": pd.Series((pi**2), index=list(range(4)), dtype=decimal_type),
    }

)

df_arrow

,A,B,C,D,E,F,G
0,1.0,2013-01-02,1.0,3,test,foo,9.87
1,1.0,2013-01-02,1.0,3,train,foo,9.87
2,1.0,2013-01-02,1.0,3,test,foo,9.87
3,1.0,2013-01-02,1.0,3,train,foo,9.87


In [9]:
#df_arrow.dtypes
df_arrow.memory_usage()

Index     32
A         32
B         32
C          9
D          8
E        128
F         32
G         65
dtype: int64

#### Leitura

1. Arquivos CSV


**Parâmetros**
- encoding
- engine 
- header
- sep
- low_memory
- dtype_backend
- chunksize
- iterator

Ao realizar a leitura de arquivos pode ser informada a engine que fará a leitura e avaliação dos dados

In [10]:
data = io.StringIO("""a,b,c
   1,2.5,True
   3,4.5,False
""")

df_csv = pd.read_csv(data, engine="pyarrow")
df_csv.dtypes

a      int64
b    float64
c       bool
dtype: object

In [11]:
data = io.StringIO("""a,b,c
   1,2.5,True
   3,4.5,False
""")

df_arrow_csv = pd.read_csv(data, dtype_backend="pyarrow", engine="pyarrow")
df_arrow_csv.dtypes

a     int64[pyarrow]
b    double[pyarrow]
c      bool[pyarrow]
dtype: object

2. Arquivos Parquet

- Engines:
- Tipo nuláveis:
- StorageOptions (Opcional):

```python
from io import BytesIO

original_df = pd.DataFrame(
    {"foo": range(5), "bar": range(5, 10)}
   )

df_parquet_bytes = original_df.to_parquet()
restored_df = pd.read_parquet(BytesIO(df_parquet_bytes))
restored_df
```

In [5]:
from io import BytesIO

original_df = pd.DataFrame(
    {"foo": range(5), "bar": range(5, 10)}
   )

df_parquet_bytes = original_df.to_parquet()
restored_df = pd.read_parquet(BytesIO(df_parquet_bytes), engine="pyarrow")
restored_df

,foo,bar
0,0,5
1,1,6
2,2,7
3,3,8
4,4,9


##### Semântica de Acesso

Por semântica de acesso, vamos definir as formas de acessar as componentes do DataFrame pandas.

- Seleção baseada em indíce 
```python
base_semana02.iloc[:, 0]
```

- Seleção baseada em Slices

```python
base_semana02.iloc[-5:]
pandas_df.iloc[lambda x: x.index % 2 == 0]
pandas_df.iloc[1:10:5, 1:100:2]
```

- Seleção baseada em rótulos
```python
base_semana02.loc[0, 'id']
```

##### Manipulação de indíces
```python
base_semana02.set_index("id")
```


##### Seleção condicional
```python
base_semana02.loc[base_semana02.price.notnull()]
base_semana02.loc[base_semana02.var_1.isin([2, 3])]
base_semana02.loc[base_semana02.var_1 > 2]

```

##### Atribuições
```python
base_semana02['amostra'] = 'teste'
```

#### Sumários e agregações 

```python

# Análise descritiva simplificada
review_points_mean = reviews.points.mean()

# Aplicação de uma função anônima para o computo de agregações mais complexas
reviews.points.map(lambda p: p - review_points_mean)

# Aplicação de uma função para o computo de agregações mais complexas
def remean_points(row):
    row.points = row.points - review_points_mean
    return row

reviews.apply(remean_points, axis='columns')
``` 

#### Agrupamentos e ordenação

```python
lista_colunas = []

reviews.groupby('coluna')
reviews.groupby('coluna').price.min()
reviews.groupby('coluna').apply(lambda df: df.title.iloc[0])
reviews.groupby(lista_colunas).price.agg([len, min, max])
```

In [ ]:
countries_reviewed = reviews.groupby(['country', 'province']).description.agg([len])

#### Ordenação

In [ ]:
countries_reviewed.sort_values(by='len')
countries_reviewed.sort_values(by='len', ascending=False)
countries_reviewed.sort_values(by=['country', 'len'])

#### Combinando DataFrames 

- concat
- join 
- merge

### Dask

In [14]:
import dask.dataframe as dd
from dask.distributed import Client

client = Client(n_workers=1, threads_per_worker=2, processes=True, memory_limit='4GB')
client

c:\Users\RenatoFileto\curso_bigdata\venv\lib\site-packages\distributed\node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56794 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:56794/status,
Dashboard: http://127.0.0.1:56794/status,Workers: 1
Total threads: 2,Total memory: 3.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56795,Workers: 1
Dashboard: http://127.0.0.1:56794/status,Total threads: 2
Started: Just now,Total memory: 3.73 GiB
Comm: tcp://127.0.0.1:56803,Total threads: 2
Dashboard: http://127.0.0.1:56804/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:56798,


In [17]:
from pathlib import Path

caminho_base = Path("C:/Users/RenatoFileto/curso_bigdata/ModuloPython/data/raw")

df = dd.read_csv(caminho_base.joinpath('base_semana02.csv'), sep=";", assume_missing=True)
# df.head(10)

df.head(10)

,id,amostra,foto,conceito,var_1,var_2,var_3,var_4,var_5,var_6,...,var_483,var_484,var_485,var_486,var_487,var_488,var_489,var_490,var_491,var_492
0,10000125.0,OOT,set22,0.0,45.0,45.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000177.0,DES,mai22,0.0,79.0,79.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10000360.0,OOT,set22,0.0,83.0,83.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10001297.0,DES,jul22,0.0,81.0,81.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10002398.0,DES,dez21,0.0,74.0,74.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1000261.0,TTD,fev23,NaN,172.0,200.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1000357.0,OOS,jul22,0.0,192.0,193.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,10005229.0,OOS,dez21,0.0,74.0,74.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,10005447.0,DES,fev22,0.0,27.0,27.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10005777.0,TTD,fev23,NaN,17.0,17.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Referências

1. **Pandas Workout** - Reuven M. Lerner, Manning Publications
2. **Python Data Analytics** - Fabio Nelli, Apress
3. **Modern Pandas Tutorial** - [Tom's Blog](https://tomaugspurger.net/posts/modern-1-intro/)
4. **Practical Pandas** - [Tom's Blog](https://tomaugspurger.net/posts/pp01/)